# VGG16-54 R1 R4 — proof-bound resume through epoch 2

Train/validation only. Run on a genuinely new NVIDIA L4 after the independent fresh artifact QA proof has been uploaded. This notebook rehashes the complete epoch-1 state before resume and stops exactly at epoch 2. Never open the sealed test.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/M1_D1_Colab')
ARCHIVE = DRIVE / 'cascade_b0_r1.tar.gz'
ARCHIVE_SHA256 = '5a5a9afbf06b9bc2f55546f5a49738ed9ca9ac6f96a78931f78669e71dab4806'
MANIFEST_SHA256 = 'bae147b6f7c6f80ac973923df4b60fd31b6587221fe75eaa2d9ed4b93ee6a51b'
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
CHECKOUT_COMMIT = '2e1d981e70e02b19d1998fa17be8c8a0de033df0'
OWNER_APPROVAL = DRIVE / 'vgg16_54_r1_owner_approval.json'
OWNER_APPROVAL_SHA256 = 'f09c7707263b52337957de5dd9ac2bf22712bb9135df8a84880633718ade06af'
SOURCE_QA = DRIVE / 'vgg16_54_r1_source_qa_proof.json'
SOURCE_QA_SHA256 = '10f0ca69e32aed906d0bf8cad36aaacf3d40ea069faec51fb084de0d7ecfb216'
RUNTIME_SOURCE_QA = DRIVE / 'vgg16_54_r1_runtime_r1_source_qa_proof.json'
RUNTIME_SOURCE_QA_SHA256 = 'cbd531302429908354858640ac501e8c0d1805c5dec5462bf5e6a1e611b044f5'
FRESH_SMOKE_QA = DRIVE / 'vgg16_54_r1_r3_fresh_smoke_qa_proof.json'
FRESH_SMOKE_QA_SHA256 = '97b9dbbba6ebb99e30df733949684e9aef2e486f7e8357cee4b90968e42da580'
SMOKE_QA = None
SMOKE_QA_SHA256 = None
MODE = 'resume'
STOP_AFTER = 2
STATE = DRIVE / 'runs/vgg16_54_seed_20261018_r2'
ROOT = Path('/content/vgg16_54_input')
REPO = Path('/content/Traffic_Sign_Detector')


In [ ]:
import hashlib
import json
import os
import stat
import subprocess

def sha256(path):
    value = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

assert len(CHECKOUT_COMMIT) == 40 and all(
    c in '0123456789abcdef' for c in CHECKOUT_COMMIT
)
for path, value in (
    (OWNER_APPROVAL, OWNER_APPROVAL_SHA256),
    (SOURCE_QA, SOURCE_QA_SHA256),
    (RUNTIME_SOURCE_QA, RUNTIME_SOURCE_QA_SHA256),
    (FRESH_SMOKE_QA, FRESH_SMOKE_QA_SHA256),
):
    assert len(value) == 64 and all(c in '0123456789abcdef' for c in value)
    assert sha256(path) == value
assert sha256(ARCHIVE) == ARCHIVE_SHA256
subprocess.run(['nvidia-smi'], check=True)

def reject_constant(value):
    raise ValueError(f'non-finite JSON constant: {value}')

def unique_object(pairs):
    result = {}
    for key, value in pairs:
        assert key not in result, f'duplicate JSON key: {key}'
        result[key] = value
    return result

def strict_json(path):
    return json.loads(
        path.read_text(), parse_constant=reject_constant,
        object_pairs_hook=unique_object,
    )

def safe_file(path):
    info = path.lstat()
    assert stat.S_ISREG(info.st_mode) and info.st_nlink == 1
    return info

proof = strict_json(FRESH_SMOKE_QA)
assert set(proof) == {
    'schema_version', 'verdict', 'package', 'state_suffix', 'source_commit',
    'launcher_commit', 'launch_notebook_sha256', 'launch_qa_record_sha256',
    'config_sha256', 'bundle_manifest_sha256', 'status', 'event', 'epoch',
    'attempt_count', 'fresh_attempt', 'runtime_identity', 'gpu',
    'run_binding_sha256', 'checkpoint_binding_sha256', 'checkpoint_generation',
    'checkpoint_manifest_sha256', 'snapshot_files', 'state_artifacts',
    'sealed_test_accessed', 'authorization',
}
assert proof['schema_version'] == 1 and proof['verdict'] == 'PASS'
assert proof['package'] == 'VGG16-54-R1-R3-FRESH-SMOKE'
assert proof['state_suffix'] == 'M1_D1_Colab/runs/vgg16_54_seed_20261018_r2'
assert proof['source_commit'] == CHECKOUT_COMMIT
assert proof['config_sha256'] == 'b5275ca6635859f3eb2ae5d98e480da18f748667a96f92a36d1e2145b197c15a'
assert proof['bundle_manifest_sha256'] == MANIFEST_SHA256
assert (proof['status'], proof['event'], proof['epoch']) == ('SMOKE_STOP', 'SMOKE_STOP', 1)
assert proof['attempt_count'] == 1
assert proof['fresh_attempt'] == {'mode': 'fresh', 'completed_epoch': 0}
assert proof['runtime_identity'] == {
    'schema': 'sha256:d1-runtime-v1:tag+linux-boot-id',
    'digest': '15f95cca74680341a2e90aeee1e6679050ddf3974ab525e6ad7366eb1000453f',
}
assert proof['gpu']['device_name'] == 'NVIDIA L4'
assert proof['run_binding_sha256'] == proof['checkpoint_binding_sha256']
assert proof['run_binding_sha256'] == 'fe334b1877454adb608628bde0c1a3287a31b861f5bb7b16635e96a8a1953bbe'
assert proof['checkpoint_generation'] == 'epoch-0001'
assert proof['sealed_test_accessed'] is False
assert proof['authorization'] == 'RESUME_EPOCH_2_ON_GENUINELY_NEW_L4_ONLY'
expected_state = {
    'colab_execution.json', 'DB_logger.db', 'train.log',
    'recovery/checkpoint_io.jsonl', 'recovery/latest.json',
}
assert set(proof['state_artifacts']) == expected_state
for relative, record in proof['state_artifacts'].items():
    assert set(record) == {'size', 'sha256'}
    artifact = STATE / relative
    assert safe_file(artifact).st_size == record['size']
    assert sha256(artifact) == record['sha256']
snapshot = STATE / 'recovery/snapshots' / proof['checkpoint_generation']
rows = {row['path']: row for row in proof['snapshot_files']}
assert set(rows) == {
    'weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml',
    'run_contract.json',
}
for relative, record in rows.items():
    assert set(record) == {'path', 'size', 'sha256'}
    artifact = snapshot / relative
    assert safe_file(artifact).st_size == record['size']
    assert sha256(artifact) == record['sha256']
manifest_path = snapshot / 'checkpoint_manifest.json'
safe_file(manifest_path)
assert sha256(manifest_path) == proof['checkpoint_manifest_sha256']
roster = {str(path.relative_to(STATE)) for path in STATE.rglob('*') if path.is_file()}
expected_roster = expected_state | {
    f'recovery/snapshots/{proof["checkpoint_generation"]}/{relative}'
    for relative in rows
} | {f'recovery/snapshots/{proof["checkpoint_generation"]}/checkpoint_manifest.json'}
assert roster == expected_roster
execution = strict_json(STATE / 'colab_execution.json')
assert execution['execution_attempts'][0]['runtime_identity'] == proof['runtime_identity']
assert execution['execution_attempts'][0]['gpu'] == proof['gpu']
assert execution['sealed_test_accessed'] is False


In [ ]:
import subprocess

assert not REPO.exists(), 'Use a fresh runtime checkout'
subprocess.run(
    ['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True
)
subprocess.run(
    ['git', 'checkout', '--detach', CHECKOUT_COMMIT], cwd=REPO, check=True
)
observed = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()
assert observed == CHECKOUT_COMMIT
subprocess.run(['pip', 'install', '-q', 'uv==0.11.14'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=REPO / 'detector', check=True)


In [ ]:
import subprocess
import sys

assert not ROOT.exists(), 'Use a fresh extraction destination'
subprocess.run([
    sys.executable, str(REPO / 'detector/colab/extract_cascade_b0_r1.py'),
    str(ARCHIVE), str(ROOT), '--archive-sha256', ARCHIVE_SHA256,
    '--manifest-sha256', MANIFEST_SHA256,
], check=True)


In [ ]:
import subprocess

command = [
    'uv', 'run', 'python', '-m', 'colab.run_vgg16_54_r1_r1',
    '--root', str(ROOT), '--state', str(STATE),
    '--bundle-manifest-sha256', MANIFEST_SHA256,
    '--owner-approval', str(OWNER_APPROVAL),
    '--owner-approval-sha256', OWNER_APPROVAL_SHA256,
    '--source-qa', str(SOURCE_QA), '--source-qa-sha256', SOURCE_QA_SHA256,
    '--runtime-source-qa', str(RUNTIME_SOURCE_QA),
    '--runtime-source-qa-sha256', RUNTIME_SOURCE_QA_SHA256,
    '--mode', MODE,
]
if STOP_AFTER is not None:
    command += ['--stop-after', str(STOP_AFTER)]
else:
    command += [
        '--smoke-qa', str(SMOKE_QA),
        '--smoke-qa-sha256', SMOKE_QA_SHA256,
    ]
subprocess.run(command, cwd=REPO / 'detector', check=True)


In [ ]:
import json
import sqlite3

execution = json.loads((STATE / 'colab_execution.json').read_text())
print(json.dumps(execution, indent=2, ensure_ascii=False))
with sqlite3.connect(STATE / 'DB_logger.db') as database:
    rows = list(database.execute(
        'SELECT time, stage, event, payload FROM logs ORDER BY rowid DESC LIMIT 20'
    ))
for row in rows:
    print(row)
attempts = execution['execution_attempts']
assert execution['status'] == 'SMOKE_STOP' and execution['epoch'] == 2
assert execution['sealed_test_accessed'] is False
assert len(attempts) == 2
assert attempts[0]['mode'] == 'fresh' and attempts[0]['completed_epoch'] == 0
assert attempts[1]['mode'] == 'resume' and attempts[1]['completed_epoch'] == 1
assert attempts[1]['gpu']['device_name'] == 'NVIDIA L4'
assert attempts[0]['runtime_identity']['digest'] != attempts[1]['runtime_identity']['digest']
assert attempts[1]['run_binding_sha256'] == attempts[1]['checkpoint_binding_sha256']


## Gate after this notebook

Stop after epoch 2. Independent two-runtime artifact QA must verify both runtime digests, the epoch-2 checkpoint, all three logs and `sealed_test_accessed=false`. Long training remains forbidden until that QA produces a separately pinned proof.
